# Training EfficientNet-B4 pada Dataset DFDC
Notebook ini melatih model EfficientNet-B4 pretrained ImageNet pada subset
train sample dataset DeepFake Detection Challenge (DFDC), sebagai bagian dari
penelitian evaluasi generalisasi deteksi deepfake pada konteks video politik Indonesia.

In [ ]:
!pip install kaggle --quiet

In [ ]:
!kaggle datasets download -d itamargr/dfdc-faces-of-the-train-sample

Dataset URL: https://www.kaggle.com/datasets/itamargr/dfdc-faces-of-the-train-sample
License(s): ODbL-1.0
100% 3.64G/3.64G [00:48<00:00, 80.8MB/s]



In [ ]:
!unzip -q dfdc-faces-of-the-train-sample.zip -d dfdc_faces/

In [ ]:
import os

for root, dirs, files in os.walk('dfdc_faces'):
    level = root.replace('dfdc_faces', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level >= 2:  # 2 level folder
        continue

# Hitung jumlah file per kelas
train_fake = len(os.listdir('dfdc_faces/train/fake'))
train_real = len(os.listdir('dfdc_faces/train/real'))
print(f"\nTrain FAKE: {train_fake} gambar")
print(f"Train REAL: {train_real} gambar")

dfdc_faces/
  validation/
    real/
    fake/
  train/
    real/
    fake/

Train FAKE: 73154 gambar
Train REAL: 20699 gambar


In [ ]:
!pip install timm scikit-learn --quiet

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import os
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder khusus untuk simpan model
SAVE_DIR = '/content/drive/MyDrive/gemastik_deepfake'
os.makedirs(SAVE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class FaceDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.samples = []

        for fname in os.listdir(real_dir):
            self.samples.append((os.path.join(real_dir, fname), 0))  # label 0 = REAL

        for fname in os.listdir(fake_dir):
            self.samples.append((os.path.join(fake_dir, fname), 1))  # label 1 = FAKE

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # standar ImageNet
])

train_dataset = FaceDataset(
    real_dir='dfdc_faces/train/real',
    fake_dir='dfdc_faces/train/fake',
    transform=transform
)

val_dataset = FaceDataset(
    real_dir='dfdc_faces/validation/real',
    fake_dir='dfdc_faces/validation/fake',
    transform=transform
)

print(f"Total training: {len(train_dataset)} gambar")
print(f"Total validation: {len(val_dataset)} gambar")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

Total training: 93853 gambar
Total validation: 30794 gambar


In [ ]:
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.safetensors: reconstructing file:   0%|          |  0.00B / 77.9MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
EPOCHS = 1

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), f'{SAVE_DIR}/efficientnet_b4_epoch{epoch+1}.pth')

print("Training selesai!")

Epoch 1/1 - Loss: 0.1373
Training selesai!


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
cm = confusion_matrix(all_labels, all_preds)

print(f"Akurasi: {acc:.4f}")
print(f"Presisi: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Confusion Matrix:\n{cm}")

Akurasi: 0.8657
Presisi: 0.9451
Recall: 0.8844
F1-Score: 0.9137
Confusion Matrix:
[[ 4756  1273]
 [ 2864 21901]]


In [ ]:
torch.save(model.state_dict(), f'{SAVE_DIR}/model_final_efficientnet_b4.pth')

with open(f'{SAVE_DIR}/hasil_skenario_A.txt', 'w') as f:
    f.write(f"Akurasi: {acc:.4f}\nPresisi: {precision:.4f}\nRecall: {recall:.4f}\nF1: {f1:.4f}\nConfusion Matrix:\n{cm}")

print("Model final tersimpan")

Model final tersimpan
Model final tersimpan
